# Oficina de Pré-processamento de Dados
## Biomedicina e Farmácia

### Simulação: amostras laboratoriais didáticas

Os dados deste notebook são fictícios. Não representam pacientes, exames reais ou resultados clínicos.

**Objetivos:** identificar problemas, tratar dados ausentes, padronizar textos, validar números, detectar possíveis outliers, normalizar variáveis, discretizar dados e interpretar gráficos.

## Identificação dos grupos

Cada grupo deve ter pelo menos quatro estudantes. Preencham os nomes antes de começar.

### Grupo 1 - Diagnóstico
1. __________________________________  2. __________________________________
3. __________________________________  4. __________________________________

### Grupo 2 - Limpeza e padronização
1. __________________________________  2. __________________________________
3. __________________________________  4. __________________________________

### Grupo 3 - Dados numéricos
1. __________________________________  2. __________________________________
3. __________________________________  4. __________________________________

### Grupo 4 - Transformação e gráficos
1. __________________________________  2. __________________________________
3. __________________________________  4. __________________________________

**Sugestão de funções:** leitura da explicação, execução do código, observação do resultado e registro da conclusão.

## Organização da atividade

Cada grupo executará uma etapa e explicará à turma o que o código fez.

| Grupo | Etapa | Pergunta |
|---|---|---|
| 1 | Diagnóstico | Que problemas existem? |
| 2 | Limpeza | Como padronizar e tratar ausências? |
| 3 | Números | Como validar valores e detectar suspeitas? |
| 4 | Transformação | Como preparar e visualizar os dados? |

Uma decisão em uma etapa interfere nas seguintes. Por isso, todos devem acompanhar o notebook inteiro.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import unicodedata
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## Base bruta

Um laboratório didático reuniu informações de amostras para estudar a qualidade de uma preparação. Durante a coleta, foram inseridos problemas comuns em bases reais: ausências, grafias diferentes, texto em campo numérico, pH suspeito, temperatura discrepante e duplicidade.

In [ ]:
dados = {
    'amostra_id': ['A01','A02','A03','A04','A05','A05','A06','A07','A08','A09','A10','A11'],
    'material': ['Sangue','urina ','Plasma','SANGUE','Urína','Urína','Saliva',None,'plasma','Sangue','SALIVA','Saliva'],
    'concentracao_mg_ml': ['2.5','1.8',None,'3,2','2.1','2.1','1.5','4.0','2.8','2.7','1.2','1.9'],
    'ph': [7.4,6.2,7.1,7.8,None,6.8,7.0,14.0,7.3,7.5,6.0,6.7],
    'temperatura_c': [36.8,37.1,36.5,38.0,36.9,36.9,37.2,36.7,80.0,36.6,37.0,None],
    'origem': ['Nova Iguaçu','nova iguaçu','Rio de Janeiro','RIO DE JANEIRO','Niterói','Niterói','nova iguaçu','Duque de Caxias','Rio de Janeiro','Niterói','duque de caxias','Nova Iguaçu'],
    'resultado': ['Adequado','adequado','Inadequado','ADEQUADO','Adequado','Adequado',None,'Inadequado','Adequado','adequado','Inadequado','Adequado']
}
df = pd.DataFrame(dados)
display(df)

## GRUPO 1 - Diagnóstico

Antes de corrigir, observe. O diagnóstico evita alterações sem justificativa.

In [ ]:
print('Dimensões:', df.shape)
print('\nTipos de dados:')
display(df.dtypes.to_frame('tipo'))
print('\nAusências:')
display(df.isna().sum().to_frame('quantidade'))

In [ ]:
for coluna in ['material','origem','resultado']:
    print(f'\n{coluna}:', df[coluna].dropna().unique())

### Registro do Grupo 1

**Problemas encontrados:**

____________________________________________________________________

____________________________________________________________________

**Que dado deveria ser confirmado na fonte original?**

____________________________________________________________________

## GRUPO 2 - Limpeza e padronização

Informações equivalentes precisam ser reconhecidas como equivalentes. Vamos remover espaços, acentos e diferenças entre maiúsculas e minúsculas.

In [ ]:
def remover_acentos(texto):
    if pd.isna(texto):
        return texto
    texto = unicodedata.normalize('NFKD', str(texto))
    return ''.join(c for c in texto if not unicodedata.combining(c))

for coluna in ['material','origem','resultado']:
    df[coluna] = df[coluna].apply(remover_acentos)
    df[coluna] = df[coluna].astype('string').str.strip().str.upper()

df['concentracao_mg_ml'] = df['concentracao_mg_ml'].astype('string').str.replace(',', '.', regex=False)
df['concentracao_mg_ml'] = pd.to_numeric(df['concentracao_mg_ml'], errors='coerce')

display(df)

In [ ]:
df['ph'] = df['ph'].fillna(df['ph'].median())
df['temperatura_c'] = df['temperatura_c'].fillna(df['temperatura_c'].median())
df['material'] = df['material'].fillna('NAO INFORMADO')
df['concentracao_mg_ml'] = df['concentracao_mg_ml'].fillna(df['concentracao_mg_ml'].median())
df['resultado'] = df['resultado'].fillna('NAO INFORMADO')

print('Ausências depois do tratamento:')
display(df.isna().sum().to_frame('quantidade'))

### Registro do Grupo 2

**Por que não devemos preencher todo valor ausente com zero?**

____________________________________________________________________

**Qual foi a vantagem da padronização?**

____________________________________________________________________

## GRUPO 3 - Validação numérica e possíveis outliers

Um valor discrepante não é automaticamente um erro. Ele pode ser uma situação real, um erro de digitação ou um problema de medição. Primeiro sinalizamos; depois investigamos.

In [ ]:
print('pH fora do intervalo didático:')
display(df[~df['ph'].between(0, 10)])

print('Temperatura suspeita:')
display(df[~df['temperatura_c'].between(30, 45)])

print('Possíveis duplicidades:')
display(df[df.duplicated('amostra_id', keep=False)])

In [ ]:
# Decisão didática: confirmar, corrigir ou marcar para investigação.
ph_validos = df.loc[df['ph'].between(0, 10), 'ph']
df.loc[~df['ph'].between(0, 10), 'ph'] = ph_validos.median()

df.loc[~df['temperatura_c'].between(30, 45), 'temperatura_c'] = None
df['temperatura_c'] = df['temperatura_c'].fillna(df['temperatura_c'].median())

df = df.drop_duplicates(subset='amostra_id', keep='first').reset_index(drop=True)
display(df)

### Registro do Grupo 3

**Por que um outlier não deve ser apagado automaticamente?**

____________________________________________________________________

**Que documento ou pessoa poderia confirmar uma medição suspeita?**

____________________________________________________________________

## GRUPO 4 - Normalização e discretização

A normalização coloca uma variável em outra escala. A discretização transforma números em categorias.

In [ ]:
df['concentracao_normalizada'] = (
    (df['concentracao_mg_ml'] - df['concentracao_mg_ml'].min()) /
    (df['concentracao_mg_ml'].max() - df['concentracao_mg_ml'].min())
)

def classificar_ph(valor):
    if valor < 6.5:
        return 'ACIDO'
    elif valor <= 7.5:
        return 'PROXIMO_DA_NEUTRALIDADE'
    return 'ALCALINO'

df['categoria_ph'] = df['ph'].apply(classificar_ph)
display(df[['amostra_id','concentracao_mg_ml','concentracao_normalizada','ph','categoria_ph']])

### Gráfico de barras

Usamos barras para comparar categorias, como a quantidade de cada tipo de material.

In [ ]:
df['material'].value_counts().plot(kind='bar', color='#4C78A8')
plt.title('Quantidade de amostras por material')
plt.xlabel('Material')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Boxplot

O boxplot permite observar a distribuição dos valores e possíveis valores discrepantes.

In [ ]:
df[['concentracao_mg_ml','ph','temperatura_c']].plot(kind='box')
plt.title('Distribuição das variáveis numéricas')
plt.ylabel('Valores')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Gráfico de dispersão

O gráfico de dispersão compara duas variáveis numéricas. Uma relação visual não prova causa e efeito.

In [ ]:
df.plot(x='concentracao_mg_ml', y='ph', kind='scatter', color='#7A5195', s=90)
plt.title('Relação entre concentração e pH')
plt.xlabel('Concentração (mg/mL)')
plt.ylabel('pH')
plt.tight_layout()
plt.show()

## Conferência final

Pré-processar não é esconder problemas. É identificar, documentar e tratar os dados com justificativa.

In [ ]:
print('Formato final:', df.shape)
print('Ausências restantes:')
display(df.isna().sum().to_frame('quantidade'))
display(df)

## Conclusão individual

1. Diferencie dado ausente, inconsistente e ruído.
2. Por que a concentração precisou ser convertida para número?
3. Qual técnica criou a categoria do pH?
4. Qual gráfico foi mais útil para comparar materiais?
5. Que decisão deveria ser confirmada em um laboratório real?
6. Por que uma base mal preparada prejudica análises e modelos de inteligência artificial?

**Frase do grupo:**

Uma base de dados de qualidade é importante porque ________________________________________________.

## Conceitos revisados

| Técnica | Aplicação |
|---|---|
| Inspeção | observar linhas, colunas e tipos |
| Limpeza | tratar ausências e duplicidades |
| Padronização | uniformizar textos |
| Conversão | transformar texto numérico em número |
| Validação | sinalizar valores suspeitos |
| Normalização | transformar concentração para 0 a 1 |
| Discretização | transformar pH em categorias |
| Visualização | barras, boxplot e dispersão |